# Neural TSP - two_moons OOD test (n=15)

Tests whether the **n=15 RL policy** generalizes to a point distribution it never saw:
**two_moons** (two interleaving half-circles, a non-convex geometry unlike
uniform/clustered/grid/ring) versus **uniform** (the training distribution, in-distribution
control).

Per the project convention the neural method is **Active Search** (per-instance, ~5000 gradient
steps). The metric is the **gap to 2-Opt**: `(Active Search - 2-Opt) / 2-Opt`, computed
instance-by-instance so we get a mean +/- std. 2-Opt has no training distribution, so a gap that
*widens* on two_moons (relative to the uniform control) is the signal that the policy learned
uniform-specific structure rather than general TSP skill.

**Prerequisite**: the main notebook has produced `actor.pt` (n=15) in Colab `python/` or in
Drive at `MyDrive/neural-tsp/python/actor.pt`.

Runtime: ~2 h on a T4 (50 instances x 2 distributions x 5000 active-search steps). Lower
`NUM_STEPS` or `N` in cell 3 for a faster check.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 - Copy project & install deps

In [ ]:
import os

!rm -rf /content/neural-tsp
!cp -r /content/drive/MyDrive/neural-tsp /content/neural-tsp
%cd /content/neural-tsp

!pip install torch numpy matplotlib pandas -q

assert os.path.isdir('/content/neural-tsp/variation'), 'variation/ missing from the Drive copy'
print('variation/ contents:', sorted(os.listdir('/content/neural-tsp/variation')))

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (will run on CPU)')

## 2 - Ensure the trained n=15 `actor.pt` is present

In [ ]:
import os, shutil

actor_local  = '/content/neural-tsp/python/actor.pt'
actor_drive  = '/content/drive/MyDrive/neural-tsp/python/actor.pt'

if not os.path.exists(actor_local):
    if os.path.exists(actor_drive):
        shutil.copy(actor_drive, actor_local)
        print('Copied actor.pt from Drive -> python/actor.pt')
    else:
        raise FileNotFoundError('actor.pt not found - run the main training notebook first.')
else:
    print('actor.pt already present in python/')
print('actor.pt ready:', os.path.exists(actor_local))

## 3 - Config + generate uniform (control) & two_moons at n=15

`uniform` is the in-distribution control; `two_moons` is the OOD test. Active Search is
expensive, so N is small (50). Edit `N` / `NUM_STEPS` to trade speed for precision.

In [ ]:
import sys, os, numpy as np, torch
sys.path.insert(0, '/content/neural-tsp/variation')
sys.path.insert(0, '/content/neural-tsp/python')

import generate_ood as G

# ---- knobs ----
N          = 50        # instances per distribution (active search is expensive)
GRAPH_N    = 15        # MUST match the trained actor.pt
NUM_STEPS  = 5000      # active-search gradient steps per instance
BATCH      = 128
AS_LR      = 1e-5
DISTS      = ['uniform', 'two_moons']   # uniform = in-distribution control
# ----------------

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

DATA_DIR = '/content/neural-tsp/variation/data'
os.makedirs(DATA_DIR, exist_ok=True)
paths = {}
for name in DISTS:
    p = os.path.join(DATA_DIR, f'{name}.txt')
    pts = G.generate_distribution(name, N, GRAPH_N, seed=0)
    G.write_tsp_file(p, pts)
    G.validate(p, N, GRAPH_N)
    paths[name] = p
    print(f'  generated {name:<10} {pts.shape}  mean_NN_dist={G.mean_nn_distance(pts):.4f}')
print('datasets ready.')

## 4 - Run NN/2-Opt (C++) + Active Search (neural), per instance

NN and 2-Opt come from the compiled C++ `baselines --per-instance` (the canonical
implementation). Active Search reloads `actor.pt` fresh each instance (it mutates weights).

In [ ]:
import subprocess, io, contextlib
import pandas as pd
from run_study import ensure_baselines
from dataset_rl import TSPDatasetRL
from active_search import active_search

binp = ensure_baselines()

def baselines_per_instance(path):
    with open(path) as fh:
        out = subprocess.run([binp, '--per-instance'], stdin=fh, capture_output=True, text=True, check=True)
    nn, two = [], []
    for line in out.stdout.splitlines():
        if line.startswith('PI '):
            _, _, a, b = line.split()
            nn.append(float(a)); two.append(float(b))
    return np.array(nn), np.array(two)

rows = []
sample = {}   # name -> (points_tensor, active_search_tour)
for name in DISTS:
    nn_lens, two_lens = baselines_per_instance(paths[name])
    ds = TSPDatasetRL(paths[name])
    as_lens = []
    print(f'\n[{name}] Active Search on {len(ds)} instances x {NUM_STEPS} steps ...')
    for i in range(len(ds)):
        with contextlib.redirect_stdout(io.StringIO()):
            tour, best = active_search(ds[i],
                                       state_dict_path='/content/neural-tsp/python/actor.pt',
                                       num_steps=NUM_STEPS, batch_size=BATCH, lr=AS_LR, device=device)
        as_lens.append(best)
        if i == 0:
            sample[name] = (ds[i], tour)
        print(f'  {name} {i+1}/{len(ds)}: active_search={best:.4f}  2opt={two_lens[i]:.4f}')
    as_lens, two_lens = np.array(as_lens), np.array(two_lens)
    gap = (as_lens - two_lens) / two_lens * 100
    rows.append({'distribution': name, 'N': len(ds),
                 'NN': round(nn_lens.mean(), 4),
                 '2-Opt': round(two_lens.mean(), 4),
                 'ActiveSearch': round(as_lens.mean(), 4),
                 'gap_AS_%_mean': round(gap.mean(), 2),
                 'gap_AS_%_std': round(gap.std(), 2)})

df = pd.DataFrame(rows)
df

## 5 - Gap to 2-Opt: does the policy generalize to two_moons?

Bars are `(Active Search - 2-Opt) / 2-Opt` per distribution (mean +/- std). `uniform` is the
control; a larger bar on `two_moons` = the policy degrades out-of-distribution.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
x = list(range(len(df)))
ax.bar(x, df['gap_AS_%_mean'], yerr=df['gap_AS_%_std'], capsize=6,
       color=['tab:gray', 'tab:purple'])
ax.axhline(0, color='k', linewidth=0.8)
ax.set_xticks(x); ax.set_xticklabels(df['distribution'])
ax.set_ylabel('(Active Search - 2-Opt) / 2-Opt  [%]')
ax.set_title('OOD test: Active-Search gap to 2-Opt (lower = better)')
fig.tight_layout()
fig.savefig('/content/drive/MyDrive/neural-tsp/two_moons_gap.png', dpi=150)
plt.show()

## 6 - Visual check: Active-Search tour vs 2-Opt on a two_moons instance

A qualitative look - does the policy's tour respect the two-moon structure, or does it
cross between the moons unnecessarily?

In [ ]:
import matplotlib.pyplot as plt
import torch
import sys
sys.path.insert(0, '/content/neural-tsp/visualization')
from heuristics import nearest_neighbor, two_opt

pts, as_tour = sample['two_moons']
pts = pts.float()
as_tour = torch.as_tensor(as_tour).long()
opt2_tour = two_opt(pts, nearest_neighbor(pts))

def _plot(ax, points, tour, title):
    xy = torch.cat([points[tour], points[tour][:1]]).numpy()
    pxy = points.numpy()
    ax.scatter(pxy[:, 0], pxy[:, 1], s=40, c='black', zorder=5)
    ax.plot(xy[:, 0], xy[:, 1], linewidth=2)
    ax.set_title(title); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
_plot(axes[0], pts, as_tour, 'Active Search')
_plot(axes[1], pts, opt2_tour, '2-Opt')
fig.suptitle('two_moons instance: Active Search vs 2-Opt')
fig.tight_layout(); plt.show()

## 7 - Save results to Drive

In [ ]:
import os, shutil

out_dir = '/content/drive/MyDrive/neural-tsp/variation_results'
os.makedirs(out_dir, exist_ok=True)
df.to_csv(f'{out_dir}/two_moons_gap_table.csv', index=False)
print('Saved table ->', f'{out_dir}/two_moons_gap_table.csv')

shutil.copytree('/content/neural-tsp/variation/data', f'{out_dir}/data', dirs_exist_ok=True)
print('Backed up datasets -> variation_results/data/')

## How to read the result

- **`uniform`** is the in-distribution control - Active Search should sit close to 2-Opt here.
- **`two_moons`** is out-of-distribution. If its gap bar is markedly larger than uniform's, the
  policy has learned uniform-specific structure rather than general TSP skill.
- Absolute tour length is **not** comparable across rows (the distributions have different point
  densities); compare only the `gap_AS_%` column.

Notes
- We use **Active Search** as the neural method throughout, per the project convention. Because
  Active Search *adapts* to each instance, a small gap on two_moons means "the policy can adapt
  to moons with search", not "the fixed policy already generalizes". To test the **fixed**
  policy's raw generalization instead, swap the neural call for `greedy_decode` / `sample_tours`
  (from `search.py`) at a much larger N (e.g. 1000) - far cheaper than Active Search.
- For a fast sanity check, re-run cell 3 with `N=10`, `NUM_STEPS=1000`.